This notebook is essentailly our first step and contains data ingestion code

In [1]:
import os

In [2]:
%pwd

'd:\\projects\\Text Summarizer\\notebooks'

In [3]:
os.chdir('../')

In [4]:
%pwd
# we have to perform this task from the root folder which is our project folder

'd:\\projects\\Text Summarizer'

In [5]:
# creating "entity" for defining the structure of our data (like a return type of a function)

from dataclasses import dataclass   
from pathlib import Path
# dataclass is a decorator that is used to create classes that are primarily used to store data. It automatically generates special methods like __init__(), __repr__(), and __eq__() based on the class attributes, making it easier to create classes that are mainly used for data storage and manipulation.
@dataclass(frozen=True)   # frozen means we cannot change the value of the attributes of this class
class DataIngestionConfig:  
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [7]:
# configuration.py
from textSummarizer.constants import *  # we have defined some constants in constants.py file, we will use those constants here
from textSummarizer.utils.common import read_yaml, create_directories # done in utils' common.py file

In [11]:
# configuration.py is responsible for reading the configuration files (config.yaml and params.yaml) and creating the necessary directories for the project. It also defines a class ConfigurationManager that provides methods to access the configuration settings for different components of the project, such as data ingestion, data validation, data transformation, model training, model evaluation, and model deployment. Each method returns a specific configuration object that contains the relevant settings for that component. This allows us to centralize the configuration management and easily access the settings throughout the project.

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath) # since read yaml returns config box we can access dict as we do object method ie with a do dict.key = value
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(    #
            root_dir=config.root_dir,   # directly reading from config.yaml file and if we want to change something we will do it there and not here
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config

Config Manager is Done

Now we will create components for data ingestion. We will create a class called DataIngestion which will have a method called initiate_data_ingestion which will perform the data ingestion task.

In [12]:
import urllib.request as request        # to download data from url
import zipfile  # to unzip the data
from textSummarizer.logging import logger
from textSummarizer.utils.common import get_size

In [13]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config


    
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} download! with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")  

        
    
    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [16]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    #data_ingestion.download_file()  # download was causing an issue so manually done that
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e




[2026-03-07 14:58:02,919: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-07 14:58:02,927: INFO: common: yaml file: params.yaml loaded successfully]


[2026-03-07 14:58:02,932: INFO: common: created directory at: artifacts]
[2026-03-07 14:58:02,935: INFO: common: created directory at: artifacts/data_ingestion]
